In [2]:
!pip install -q transformers datasets accelerate

import os
import torch
from datasets import Dataset
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    TrainingArguments,
    Trainer,
    DataCollatorForLanguageModeling,
    set_seed
)

set_seed(42)

In [3]:
from google.colab import files
import os

# ── Upload your writing samples file ──
print("📤 Upload your my_stories.txt file now...")
uploaded = files.upload()

txt_filename = list(uploaded.keys())[0]

# Save to a consistent location
os.makedirs("/content/data", exist_ok=True)
os.rename(txt_filename, "/content/data/my_stories.txt")

print(f"✅ Saved to /content/data/my_stories.txt")

📤 Upload your my_stories.txt file now...


Saving my_stories.txt to my_stories.txt
✅ Saved to /content/data/my_stories.txt


In [4]:
def split_text(text, chunk_size=1000, overlap=200):
    chunks = []
    start = 0

    while start < len(text):
        end = start + chunk_size
        chunk = text[start:end]
        chunks.append(chunk)
        start += chunk_size - overlap

    return chunks

# --- Fix Start ---
with open("/content/data/my_stories.txt", "r") as f:
    text = f.read()
# --- Fix End ---

chunks = split_text(text)

print(f"✅ Total chunks: {len(chunks)}")

✅ Total chunks: 1809


In [5]:
split_idx = int(0.9 * len(chunks))

train_texts = chunks[:split_idx]
val_texts   = chunks[split_idx:]

train_dataset = Dataset.from_dict({"text": train_texts})
val_dataset   = Dataset.from_dict({"text": val_texts})

print(f"Train: {len(train_dataset)} | Val: {len(val_dataset)}")

Train: 1628 | Val: 181


In [6]:
model_name = "gpt2"

tokenizer = AutoTokenizer.from_pretrained(model_name)

# ✅ Proper PAD token (CRITICAL FIX)
tokenizer.add_special_tokens({'pad_token': '[PAD]'})

print("Pad token ID:", tokenizer.pad_token_id)

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


config.json:   0%|          | 0.00/665 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/26.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

Pad token ID: 50257


In [7]:
def tokenize_fn(example):
    return tokenizer(
        example["text"],
        truncation=True,
        padding="max_length",
        max_length=512,
        return_attention_mask=True
    )

tokenized_train = train_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])
tokenized_val   = val_dataset.map(tokenize_fn, batched=True, remove_columns=["text"])

print("✅ Tokenization complete")

Map:   0%|          | 0/1628 [00:00<?, ? examples/s]

Map:   0%|          | 0/181 [00:00<?, ? examples/s]

✅ Tokenization complete


In [8]:
model = AutoModelForCausalLM.from_pretrained(model_name)

# Resize embeddings because we added PAD token
model.resize_token_embeddings(len(tokenizer))

model.config.pad_token_id = tokenizer.pad_token_id
model.config.use_cache = False

model.safetensors:   0%|          | 0.00/548M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT2LMHeadModel LOAD REPORT from: gpt2
Key                  | Status     |  | 
---------------------+------------+--+-
h.{0...11}.attn.bias | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


generation_config.json:   0%|          | 0.00/124 [00:00<?, ?B/s]

The new embeddings will be initialized from a multivariate normal distribution that has old embeddings' mean and covariance. As described in this article: https://nlp.stanford.edu/~johnhew/vocab-expansion.html. To disable this, use `mean_resizing=False`


In [9]:
output_dir = "/content/models/story_voice_finetuned"
os.makedirs(output_dir, exist_ok=True)

training_args = TrainingArguments(
    output_dir=output_dir,

    num_train_epochs=6,
    per_device_train_batch_size=2,
    per_device_eval_batch_size=2,
    gradient_accumulation_steps=8,

    learning_rate=5e-5,
    weight_decay=0.01,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",

    logging_steps=25,
    logging_strategy="steps",

    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,

    fp16=torch.cuda.is_available(),
    gradient_checkpointing=True,

    save_total_limit=2,
    report_to="none",
)

warmup_ratio is deprecated and will be removed in v5.2. Use `warmup_steps` instead.


In [10]:
data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False
)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_train,
    eval_dataset=tokenized_val,
    data_collator=data_collator,
)

print("🚀 Training starting...")
trainer.train()

🚀 Training starting...


`loss_type=None` was set in the config but it is unrecognized. Using the default loss: `ForCausalLMLoss`.


Epoch,Training Loss,Validation Loss
1,3.546412,3.384796
2,3.333307,3.363275
3,3.228168,3.360579
4,3.156695,3.362632
5,3.103535,3.368480
6,3.101602,3.368889


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

...

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

There were missing keys in the checkpoint model loaded: ['lm_head.weight'].


TrainOutput(global_step=612, training_loss=3.2702592961928425, metrics={'train_runtime': 835.8152, 'train_samples_per_second': 11.687, 'train_steps_per_second': 0.732, 'total_flos': 2552300568576000.0, 'train_loss': 3.2702592961928425, 'epoch': 6.0})

In [11]:
trainer.save_model(output_dir)
tokenizer.save_pretrained(output_dir)

print("✅ Model saved!")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

✅ Model saved!


In [12]:
from google.colab import drive
import shutil
import os

drive.mount('/content/drive')

old_model_path = "/content/models/story_voice_finetuned"
drive_model_path = "/content/drive/MyDrive/story_voice_finetuned"

if os.path.exists(old_model_path):
    shutil.copytree(old_model_path, drive_model_path, dirs_exist_ok=True)
    print(f"✅ Model successfully copied to Google Drive at:\n{drive_model_path}")
else:
    print("❌ Model not found. Run the training again first.")

Mounted at /content/drive
✅ Model successfully copied to Google Drive at:
/content/drive/MyDrive/story_voice_finetuned


In [ ]:
from google.colab import drive
import shutil

drive.mount('/content/drive')
shutil.copytree("/content/models/story_voice_finetuned", "/content/drive/MyDrive/story_voice_finetuned", dirs_exist_ok=True)
print("✅ Model saved to Google Drive!")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
